# figure7.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from pathlib import Path
from matplotlib import font_manager
font_dir = Path(configured_path('font_dir'))
font_files = [font_dir / 'Times New Roman.ttf', font_dir / 'Times New Roman Bold.ttf']
for font_file in font_files:
    if not font_file.is_file():
        raise FileNotFoundError(font_file)
    font_manager.fontManager.addfont(str(font_file))
assert Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve() in {p.resolve() for p in font_files}
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds = ds.where(mask_ocean)
ds

In [ ]:
fdir2 = configured_path('iap_mld_file')
ds2 = xr.open_mfdataset(fdir2)

time_new = pd.date_range(
    start='1940-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds2 = ds2.assign_coords(time=time_new)

ds2 = ds2.sel(time=slice('1993','2023'))
ds2 = ds2.where(mask_ocean)
ds2

In [ ]:
fdir3 = configured_path('oras_mld_pattern')
ds3 = xr.open_mfdataset(fdir3)
ds3 = ds3.resample(time='1ME').mean(dim='time')

time_new = pd.date_range(
    start='1993-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds3 = ds3.assign_coords(time=time_new)
ds3 = ds3.where(mask_ocean)
ds3

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# ====================== 1. 分区域提取，保留lat/lon维度 ======================
# 原始数据集区域
ds_region = ds.sel(lat=slice(56, 60), lon=slice(304, 309))
# 新增两套数据区域
ds2_region = ds2.mld.sel(lat=slice(56, 60), lon=slice(304, 309))
ds3_region = ds3.MLD_sigma0_030.sel(lat=slice(56, 60), lon=slice(304, 309))

ds_region = ds_region.sel(time=slice('1993','2022'))
ds2_region = ds2_region.sel(time=slice('1993','2022'))
ds3_region = ds3_region.sel(time=slice('1993','2022'))

# ====================== 2. 先求空间平均时序（lat/lon平均） ======================
# --- 第一组：原始 Pred / True ---
pred_ts = ds_region["MLD_pred"].mean(dim=["lat", "lon"], skipna=True)
true_ts = ds_region["MLD_true"].mean(dim=["lat", "lon"], skipna=True)

# --- 第二组：ds2.mld ---
ds2_ts = ds2_region.mean(dim=["lat", "lon"], skipna=True)

# --- 第三组：ds3.MLD_sigma0_030 ---
ds3_ts = ds3_region.mean(dim=["lat", "lon"], skipna=True)

# ====================== 3. 按月分组计算气候态均值 + 多年逐月标准差 ======================
# groupby("time.month")：把所有年份同月份分到一组
# mean("time")：多年气候态平均；std("time")：该月份多年变率标准差
def calc_climatology(ds_ts):
    clim_mean = ds_ts.groupby("time.month").mean("time", skipna=True)
    clim_std  = ds_ts.groupby("time.month").std("time", skipna=True)
    return clim_mean.compute().values, clim_std.compute().values

pred_clim, pred_std = calc_climatology(pred_ts)
true_clim, true_std = calc_climatology(true_ts)
ds2_clim,  ds2_std  = calc_climatology(ds2_ts)
ds3_clim,  ds3_std  = calc_climatology(ds3_ts)

# x轴月份 1~12
months = np.arange(1, 13)

# ====================== 4. 绘图：逐月气候态季节循环 ±多年std阴影 ======================
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)

# 配色（区分4套，科研柔和配色）
colors = {
    "pred": "#d62728",
    "true": "#1f77b4",
    "ds2": "#2ca02c",
    "ds3": "#6388b4"
}

# 1. Predicted MLD
ax.plot(months, pred_clim, color=colors["pred"], lw=1.3, label="SJTU-ViT")
ax.fill_between(months, pred_clim-pred_std, pred_clim+pred_std, color=colors["pred"], alpha=0.22)

# 2. Observed True MLD
ax.plot(months, true_clim, color=colors["true"], lw=1.3, label="Obs")
ax.fill_between(months, true_clim-true_std, true_clim+true_std, color=colors["true"], alpha=0.22)

# 3. ds2 mld
ax.plot(months, ds2_clim, color=colors["ds2"], lw=1.3, label="IAP")
ax.fill_between(months, ds2_clim-ds2_std, ds2_clim+ds2_std, color=colors["ds2"], alpha=0.22)

# 4. ds3 MLD_sigma0_030
ax.plot(months, ds3_clim, color=colors["ds3"], lw=1.3, label="ORAS5")
ax.fill_between(months, ds3_clim-ds3_std, ds3_clim+ds3_std, color=colors["ds3"], alpha=0.22)

# ====================== 5. 图表美化 ======================
ax.legend(loc="upper right", frameon=True, fontsize=12)
ax.set_xlabel("Month", fontsize=14)
ax.set_ylabel("Climatological Mixed Layer Depth (m)", fontsize=14)
ax.set_title("Seasonal Cycle of MLD (56–60°N, 304–309°E)", fontsize=16)
ax.set_xticks(months)
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=14)
ax.grid(alpha=0.3, linewidth=0.4)
ax.tick_params(axis="both", labelsize=14)

plt.tight_layout()
#fig.savefig(output_dir / 'figure7_labrador_seasonal_cycle.png', dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# ====================== 区域配置：4个研究区域 ======================
region_configs = [
    {
        "name": "(a)Labrador Sea (56~60°N, 304~309°E)",
        "lat_slice": slice(56, 60),
        "lon_slice": slice(304, 309)
    },
    {
        "name": "(b)Southern Ocean (50~60°S, 230~290°E)",
        "lat_slice": slice(-60, -50),
        "lon_slice": slice(230, 290)
    },
    {
        "name": "(c)East Equatorial Pacific (5°S~5°N, 230~240°E)",
        "lat_slice": slice(-5, 5),
        "lon_slice": slice(230, 240)
    },
    {
        "name": "(d)Kuroshio Region (30~35°N, 140~145°E)",
        "lat_slice": slice(30, 35),
        "lon_slice": slice(140, 145)
    }
]

# ====================== 封装函数：单区域绘图逻辑 ======================
def get_region_clim(ds, ds2, ds3, lat_slc, lon_slc):
    # 1. 区域裁剪 + 时间筛选
    ds_r = ds.sel(lat=lat_slc, lon=lon_slc, time=slice('1993','2022'))
    ds2_r = ds2.mld.sel(lat=lat_slc, lon=lon_slc, time=slice('1993','2022'))
    ds3_r = ds3.MLD_sigma0_030.sel(lat=lat_slc, lon=lon_slc, time=slice('1993','2022'))

    # 2. 空间平均时序
    pred_ts = ds_r["MLD_pred"].mean(dim=["lat", "lon"], skipna=True)
    true_ts = ds_r["MLD_true"].mean(dim=["lat", "lon"], skipna=True)
    ds2_ts  = ds2_r.mean(dim=["lat", "lon"], skipna=True)
    ds3_ts  = ds3_r.mean(dim=["lat", "lon"], skipna=True)

    # 3. 计算逐月气候态均值+标准差
    def calc_clim(ts):
        clim_mean = ts.groupby("time.month").mean("time", skipna=True)
        clim_std  = ts.groupby("time.month").std("time", skipna=True)
        return clim_mean.compute().values, clim_std.compute().values

    pred_clim, pred_std = calc_clim(pred_ts)
    true_clim, true_std = calc_clim(true_ts)
    ds2_clim,  ds2_std  = calc_clim(ds2_ts)
    ds3_clim,  ds3_std  = calc_clim(ds3_ts)

    return pred_clim, pred_std, true_clim, true_std, ds2_clim, ds2_std, ds3_clim, ds3_std

# ====================== 绘图配置 ======================
months = np.arange(1, 13)
colors = {
    "pred": "#d62728",
    "true": "#1f77b4",
    "ds2": "#17becf",
    "ds3": "#2ca02c"
}
labels = {
    "pred": "SJTU-ViT",
    "true": "Obs",
    "ds2": "IAP",
    "ds3": "ORAS5"
}

# 创建2行2列子图
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(13.5, 10), dpi=300)
axes = axes.flatten()  # 展平为一维数组方便循环

# 循环绘制4个区域
for idx, cfg in enumerate(region_configs):
    ax = axes[idx]
    # 获取该区域气候态曲线
    pred_clim, pred_std, true_clim, true_std, ds2_clim, ds2_std, ds3_clim, ds3_std = get_region_clim(
        ds, ds2, ds3, cfg["lat_slice"], cfg["lon_slice"]
    )

    # 绘制四条曲线+标准差阴影
    ax.plot(months, pred_clim, color=colors["pred"], lw=2.8, label=labels["pred"])
    ax.fill_between(months, pred_clim-pred_std, pred_clim+pred_std, color=colors["pred"], alpha=0.08)

    ax.plot(months, true_clim, color=colors["true"], lw=2.8, label=labels["true"])
    ax.fill_between(months, true_clim-true_std, true_clim+true_std, color=colors["true"], alpha=0.08)

    ax.plot(months, ds2_clim, color=colors["ds2"], lw=2.8, label=labels["ds2"])
    ax.fill_between(months, ds2_clim-ds2_std, ds2_clim+ds2_std, color=colors["ds2"], alpha=0.08)

    ax.plot(months, ds3_clim, color=colors["ds3"], lw=2.8, label=labels["ds3"])
    ax.fill_between(months, ds3_clim-ds3_std, ds3_clim+ds3_std, color=colors["ds3"], alpha=0.08)

    # 子图美化
    ax.set_title(cfg["name"], fontsize=20, pad=8)
    ax.set_xticks(months)
    ax.set_xticklabels(["Jan"," ","Mar"," ","May"," ","Jul"," ","Sep"," ","Nov"," "], fontsize=18)
    ax.grid(alpha=0.3, linewidth=0.4)
    ax.tick_params(axis="both", labelsize=18)
    ax.set_xlim(1, 12)

    # 仅左下子图显示y轴标签，仅底部子图显示x轴标签，减少重复
    if idx in [2, 3]:
        ax.set_xlabel("Month", fontsize=19)
    if idx in [0, 2]:
        ax.set_ylabel("MLD (m)", fontsize=19)
    if idx not in [2, 3]:
        ax.set_xticklabels([])

    # 仅第一个子图放图例，避免重复遮挡
    if idx == 0:
        ax.legend(loc="upper right", frameon=True, fontsize=18)

# 600 dpi，无外边距
fig.savefig(output_dir / 'figure7.png', dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()
